# 06 — Comparación de modelos

**Protocolo:** análisis exclusivamente de archivos existentes. No se entrenan modelos ni se modifican los CSV originales. Las columnas se inspeccionan antes de utilizarlas. Las métricas no se reinterpretan automáticamente sin conocer su definición.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = Path(r"C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles")
assert ROOT.is_dir(), f"No existe carpeta: {ROOT}"
DEST = ROOT / 'analisis'
DEST.mkdir(exist_ok=True)
FILES = sorted(p for p in ROOT.rglob('*.csv') if DEST not in p.parents)
print('CSV encontrados:', len(FILES))
def load_csv(p):
    for enc in ('utf-8-sig','latin1'):
        try:
            df=pd.read_csv(p,encoding=enc,sep=None,engine='python')
            return df
        except UnicodeDecodeError: continue
    raise ValueError(f'No se pudo leer {p}')
def inventory():
    rows=[]
    for p in FILES:
        try:
            df=load_csv(p)
            rows.append(dict(archivo=str(p.relative_to(ROOT)),filas=len(df),columnas=len(df.columns),nombres_columnas=' | '.join(map(str,df.columns)),nulos=int(df.isna().sum().sum()),error=''))
        except Exception as e:
            rows.append(dict(archivo=str(p.relative_to(ROOT)),filas=None,columnas=None,nombres_columnas='',nulos=None,error=repr(e)))
    return pd.DataFrame(rows)


CSV encontrados: 86


In [2]:
inv=inventory();display(inv[['archivo','filas','nombres_columnas']]);print('Seleccione el CSV consolidado verificado; no se asumen nombres de columnas.')
candidates=[p for p in FILES if 'comparacion' in p.stem.lower() or 'metrica' in p.stem.lower()]
print('Candidatos:',*[str(p.relative_to(ROOT)) for p in candidates],sep='\n')


,archivo,filas,nombres_columnas
0,auditoria\inventario.csv,10,archivo | filas | columnas | faltantes | inici...
1,auditoria\salidas.csv,24,componente | salida
2,metricas\comparacion_deteccion.csv,189,metodo | config | archivo | n | n_alerta | fra...
3,metricas\deteccion_DT_FULL.csv,9,metodo | config | archivo | n | n_alerta | fra...
4,metricas\deteccion_DT_SIN_P1.csv,9,metodo | config | archivo | n | n_alerta | fra...
...,...,...,...
81,metricas\seleccion_XGB_FULL.csv,24,metodo | configuracion | variable | componente...
82,metricas\seleccion_XGB_SIN_P1.csv,24,metodo | configuracion | variable | componente...
83,metricas\seleccion_XGB_SIN_P2.csv,24,metodo | configuracion | variable | componente...
84,metricas\seleccion_XGB_SIN_P3.csv,24,metodo | configuracion | variable | componente...


Seleccione el CSV consolidado verificado; no se asumen nombres de columnas.
Candidatos:
metricas\comparacion_deteccion.csv


In [3]:
# Configuración explícita después de revisar el inventario.
SOURCE = next((p for p in candidates if 'comparacion_deteccion' in p.stem.lower()),None)
if SOURCE is None: raise ValueError('No se encontró comparacion_deteccion; definir SOURCE con una ruta existente.')
df=load_csv(SOURCE)
print('Origen:',SOURCE);display(df.head());print('Columnas:',df.columns.tolist());print('Tipos:',df.dtypes.to_dict())


Origen: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles\metricas\comparacion_deteccion.csv


,metodo,config,archivo,n,n_alerta,fraccion_alerta,episodios_visibles,episodios_detectados,alertas_fuera_etiqueta
0,DT,FULL,calibration,18691,94,0.005029,NaN,NaN,NaN
1,DT,FULL,test2,82770,998,0.012058,17.0,10.0,833.0
2,DT,FULL,test3,62370,1048,0.016803,10.0,7.0,440.0
3,DT,FULL,test4,129570,23217,0.179185,24.0,24.0,22190.0
4,DT,FULL,train2,201570,196305,0.973880,NaN,NaN,NaN


Columnas: ['metodo', 'config', 'archivo', 'n', 'n_alerta', 'fraccion_alerta', 'episodios_visibles', 'episodios_detectados', 'alertas_fuera_etiqueta']
Tipos: {'metodo': dtype('O'), 'config': dtype('O'), 'archivo': dtype('O'), 'n': dtype('int64'), 'n_alerta': dtype('int64'), 'fraccion_alerta': dtype('float64'), 'episodios_visibles': dtype('float64'), 'episodios_detectados': dtype('float64'), 'alertas_fuera_etiqueta': dtype('float64')}


In [4]:
# Solo columnas numéricas existentes. No se asigna orientación mejor/peor sin conocer la métrica.
num=df.select_dtypes(include='number').columns.tolist()
print('Métricas numéricas disponibles:',num)
df.to_csv(DEST/'06_tabla_original_comparacion.csv',index=False,encoding='utf-8-sig')
summary=df[num].describe().T if num else pd.DataFrame()
summary.to_csv(DEST/'06_resumen_numerico.csv',encoding='utf-8-sig');display(summary)


Métricas numéricas disponibles: ['n', 'n_alerta', 'fraccion_alerta', 'episodios_visibles', 'episodios_detectados', 'alertas_fuera_etiqueta']


,count,mean,std,min,25%,50%,75%,max
n,189.0,133783.555556,77926.524161,18691.000000,82770.000000,125970.000000,201570.000000,259170.0
n_alerta,189.0,59071.317460,65064.251127,94.000000,9474.000000,39884.000000,79622.000000,201570.0
fraccion_alerta,189.0,0.341491,0.292870,0.003666,0.105083,0.315936,0.500689,1.0
episodios_visibles,63.0,17.000000,5.761384,10.000000,10.000000,17.000000,24.000000,24.0
episodios_detectados,63.0,14.587302,6.459888,7.000000,9.000000,13.000000,22.000000,24.0
alertas_fuera_etiqueta,63.0,15431.333333,14659.073586,220.000000,5187.000000,10656.000000,22954.500000,48370.0


In [5]:
# Gráficos descriptivos, no ranking automático de modelos.
for col in num[:20]:
    fig,ax=plt.subplots(figsize=(8,4));df[col].dropna().plot(kind='hist',bins=20,ax=ax)
    ax.set_title(f'Distribución: {col}');fig.tight_layout()
    fig.savefig(DEST/f'06_distribucion_{num.index(col):02d}.png',dpi=140);plt.close(fig)
print('Generado. Para comparar por TEST y separar predicción/detección se requiere validar las columnas de identificación y la semántica de cada métrica.')


Generado. Para comparar por TEST y separar predicción/detección se requiere validar las columnas de identificación y la semántica de cada métrica.
